# Model Context Protocol (MCP)

MCP standardizes how LLMs discover and call external tools — think of it as a USB-C port for AI models. An MCP server exposes tools via a JSON schema; clients (agents) discover and invoke them. The result is modular, reusable tooling that works across frameworks.

## Implementation with Flyte v2 + the Agent harness

The earlier Flyte-v2 version *simulated* MCP: it wrapped tools as `@env.task`s and hand-wrote both the Anthropic tool schemas and the dispatch loop. Flyte v2 now speaks MCP for real. This notebook connects an `Agent` to an actual MCP server via `MCPServerSpec` — the harness performs the protocol handshake, discovers the server's tools, and surfaces them to the model transparently (with an optional `tool_prefix`).

#### ADK / FastMCP vs Flyte v2 + Agent harness

| Aspect | ADK / FastMCP | Flyte v2 + `Agent` harness |
|--------|---------------|----------------------------|
| **Connecting a server** | `MCPToolset(connection_params=...)` | `Agent(mcp_servers=[MCPServerSpec(...)])` |
| **Transport** | HTTP / SSE / stdio (npx, uvx) | `transport="stdio" / "streamable-http" / "sse"` |
| **Tool discovery** | MCP protocol | MCP protocol — handled by the harness |
| **Tool schema** | Auto from type hints | Discovered from the server, no local copy |
| **Orchestration** | `LlmAgent(tools=[MCPToolset(...)])` | `Agent.run` managed loop |
| **Secrets** | `os.environ` at import time | `flyte.Secret` injected at task execution |
| **Execution** | In-process or subprocess | Container on Kubernetes |

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' litellm fastmcp mcp

### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [ ]:
!flyte start devbox

### 2. Store your API key

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-...

### 3. Import dependencies and configure the TaskEnvironment

In [ ]:
from __future__ import annotations

import os
from datetime import timedelta

import flyte
from flyte.ai.agents import Agent, AgentResult, MCPServerSpec

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="mcp-agent", python_version=(3, 12))
    .with_pip_packages("litellm", "fastmcp>=2.0.0", "mcp")
)

mcp_env = flyte.TaskEnvironment(
    name="mcp_agent",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="1Gi"),
    secrets=[
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
)

### 4. Define a real MCP server

Instead of simulating MCP with `@env.task` tools, we define an actual MCP server with FastMCP and let the agent connect to it. The server exposes three tools — `greet`, `calculate`, `word_count` — over the standard MCP protocol. We keep its source as a module constant and materialize it to a file so the agent can launch it over **stdio** (this also ships correctly into the task's container).

In [ ]:
# A *real* MCP server, defined with FastMCP. The agent launches it over stdio and
# discovers its tools through the MCP protocol — no hand-written schema list.
SERVER_CODE = '''
from fastmcp import FastMCP

mcp = FastMCP(name="utility-tools")


@mcp.tool
def greet(name: str) -> str:
    """Generate a personalized greeting."""
    return f"Hello, {name}! Nice to meet you."


@mcp.tool
def calculate(expression: str) -> str:
    """Evaluate a simple arithmetic expression like '2 + 2 * 3'."""
    import ast
    import operator as op

    ops = {ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul,
           ast.Div: op.truediv, ast.Pow: op.pow, ast.Mod: op.mod,
           ast.FloorDiv: op.floordiv, ast.USub: op.neg, ast.UAdd: op.pos}

    def _eval(node):
        if isinstance(node, ast.Expression):
            return _eval(node.body)
        if isinstance(node, ast.Constant):
            return node.value
        if isinstance(node, ast.BinOp):
            return ops[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp):
            return ops[type(node.op)](_eval(node.operand))
        raise ValueError("unsupported expression")

    return str(_eval(ast.parse(expression, mode="eval")))


@mcp.tool
def word_count(text: str) -> str:
    """Count the number of words in a text string."""
    return f"{len(text.split())} words"


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''


def _materialize_server() -> str:
    """Write the MCP server to disk so it can be launched over stdio (works locally
    and inside the task's container — the code ships as a module constant)."""
    import pathlib
    import tempfile

    path = pathlib.Path(tempfile.gettempdir()) / "utility_mcp_server.py"
    path.write_text(SERVER_CODE)
    return str(path)


### 5. Build the agent with an MCP server

#### From simulated MCP to a real connection

The earlier version hand-wrote two things MCP is supposed to provide for free: the `MCP_TOOLS` JSON schema list, and a `_dispatch_tool` router mapping tool names back to Flyte tasks. Connecting a real `MCPServerSpec` deletes both — the harness discovers the tools from the server and routes calls automatically. We build the `Agent` inside the task so the server file is materialized in the same environment the agent runs in.

In [ ]:
@mcp_env.task(
    retries=2,
    timeout=timedelta(minutes=3),
    cache=flyte.Cache(behavior="disable"),
)
async def mcp_agent(user_query: str) -> str:
    """MCP-style agent backed by a real MCP server over stdio.

    Tool discovery and invocation are handled by the harness via the MCP protocol,
    replacing the hand-written schema list and dispatch loop.
    """
    server_path = _materialize_server()
    agent = Agent(
        name="mcp-assistant",
        model="claude-haiku-4-5",
        instructions=(
            "You are a helpful assistant with access to MCP utility tools "
            "(greet, calculate, word_count — exposed with a 'util_' prefix). "
            "Use them to answer the request, then compose a final answer."
        ),
        mcp_servers=[
            MCPServerSpec(
                name="utility",
                command=["python", server_path],
                transport="stdio",
                tool_prefix="util_",
            ),
        ],
        max_turns=8,
    )
    result: AgentResult = await agent.run.aio(user_query)
    if result.error:
        raise RuntimeError(result.error)
    return result.summary

### 7. Run locally

In [ ]:
QUERIES = [
    "Say hello to Alice and also calculate 15 * 7 + 3.",
    "How many words are in: 'The quick brown fox jumps over the lazy dog'?",
    "Greet Bob, then tell me what 100 divided by 4 is.",
]

for query in QUERIES:
    run = flyte.run(mcp_agent, user_query=query)
    run.wait()
    print(f"Query:  {query}")
    print(f"Answer: {run.outputs()[0]}")
    print("-" * 60)

### Other transports: HTTP and uvx / npx

`MCPServerSpec` speaks several transports. The example above launches a local Python server over **stdio**; you can just as easily connect to a server published as a package, or one running over **HTTP**. The agent treats all of them identically — discover, then call.

In [ ]:
# stdio via a published Python package (no local file needed) — e.g. the official time server:
stdio_uvx = MCPServerSpec(
    name="time",
    command=["uvx", "mcp-server-time"],
    transport="stdio",
    tool_prefix="time_",
)

# stdio via an npm package:
stdio_npx = MCPServerSpec(
    name="filesystem",
    command=["npx", "-y", "@modelcontextprotocol/server-filesystem", "/tmp"],
    transport="stdio",
)

# streamable-HTTP transport — point at a running MCP HTTP endpoint:
http_server = MCPServerSpec(
    name="remote-tools",
    url="http://localhost:8000/mcp",
    transport="streamable-http",
    headers={"Authorization": "Bearer <token>"},
    tool_filter=["greet", "calculate"],  # optional allow-list
)

print("Configured MCPServerSpec variants:",
      stdio_uvx.name, stdio_npx.name, http_server.name)

### Running remotely

With `ReusePolicy`, the MCP agent pods stay warm between calls — important for interactive tool-use loops where cold-starting a container (and re-handshaking the MCP server) per query would dominate latency. Each tool call discovered from the MCP server appears as a nested action under `agent.run` in the Flyte UI.

> **Note:** `ReusePolicy` is a Union-specific feature that requires a [Union deployment](https://www.union.ai/docs/v2/union/). It is not supported on the local devbox.

In [ ]:
run = flyte.run(
    mcp_agent,
    user_query="Greet Claude, calculate 42 * 8, and count the words in 'Hello World'.",
)
run.wait()
print(run.outputs()[0])